In [1]:
from sentence_transformers import (
  CrossEncoder,
  InputExample,
  losses,
  evaluation,
  SentenceTransformer
  )
from sentence_transformers.cross_encoder.evaluation import CEBinaryClassificationEvaluator
import math
import json
import torch
from torch.utils.data import DataLoader
from torch.quantization import quantize_dynamic
from datetime import datetime
import random
import math
import os

In [ ]:
data_ratio = 1

# organize the data
with open('../data/model/train.json', 'r') as f:
    train = json.load(f)
random.shuffle(train)

with open('../data/model/cv.json', 'r') as f:
    cv = json.load(f)
random.shuffle(cv)

with open('../data/model/test.json', 'r') as f:
    test = json.load(f)
random.shuffle(cv)

train_data = [InputExample(texts=x, label=y) for [x, y] in train[0:math.floor(len(train) * data_ratio)]]
cv_data = [InputExample(texts=x, label=y) for [x, y] in cv[0:math.floor(len(cv) * data_ratio)]]
test_data = [InputExample(texts=x, label=y) for [x, y] in test[0:math.floor(len(test) * data_ratio)]]

print(train_data[0].texts)

# training configs
train_batch_size = 256
eval_batch_size = 64
num_epochs = 10
warmup_steps = math.ceil(len(train) * num_epochs * 0.1)

## CrossEncoder Model - ms-marco-MiniLM-L-6-v2

In [ ]:
# initialize the model
CE_model = CrossEncoder(
  "cross-encoder/ms-marco-MiniLM-L-6-v2",
  device="mps" if torch.backends.mps.is_available() else "cpu",
  default_activation_function=torch.nn.Sigmoid()
)

CE_model_save_path = "output/training_CE_" + datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

# run the model
CE_model.fit(
    train_dataloader=DataLoader(
        dataset=train_data,
        shuffle=True,
        batch_size=train_batch_size,
        pin_memory=True),
    evaluator=CEBinaryClassificationEvaluator.from_input_examples(cv_data),
    epochs=num_epochs,
    warmup_steps=warmup_steps,
    output_path=CE_model_save_path
)

# write the configs
with open(CE_model_save_path + "/config.txt", 'w') as f:
    f.write("train batch size: " + str(train_batch_size))
    f.write("num epochs: " + str(num_epochs) + "\n")
    f.write("warmup_steps: " + str(warmup_steps) + "\n")
    f.write("data_ratio: " + str(1 / data_ratio) + "\n")

In [2]:
# ... or load already trained model from local files/ HuggingFace
CE_model = CrossEncoder('./final', default_activation_function=torch.nn.Sigmoid())
CE_model = CrossEncoder('nphach/jp-parallel-gloss', default_activation_function=torch.nn.Sigmoid())
torch.save(CE_model.model.state_dict(), "original_model.pth")
original_size = os.path.getsize("original_model.pth") / (1024 * 1024)

print(CE_model)


In [5]:
# quantize model
Q_model = CrossEncoder('nphach/jp-parallel-gloss', default_activation_function=torch.nn.Sigmoid())
Q_model.model = quantize_dynamic(
    CE_model.model,
    {torch.nn.Linear},
    dtype=torch.qint8
)
torch.save(Q_model.model.state_dict(), "quantized_model.pth")
quantized_size = os.path.getsize("quantized_model.pth") / (1024 * 1024)

print(f"original size: {original_size:.2f} mb")
print(f"quantized size: {quantized_size:.2f} mb")
print(f"reduction: {original_size - quantized_size:.2f} mb ({(1 - quantized_size/original_size)*100:.1f}%)")

print(CE_model.rank('glass',['cup', 'sleeve']))
print(Q_model.rank('glass',['cup', 'sleeve']))


original size: 86.69 mb
quantized size: 55.92 mb
reduction: 30.77 mb (35.5%)
[{'corpus_id': 0, 'score': 0.99228823}, {'corpus_id': 1, 'score': 0.9339647}]
[{'corpus_id': 0, 'score': 0.9935667}, {'corpus_id': 1, 'score': 0.9473836}]


In [ ]:
# positive cases
positive = [
  ["place of origin", "birthplace"],
  ["alien", "foreigner"],
  ["warship", "naval war vessel"],
  ["lemon treats", "lemon-flavored candy"],
  ["TikTok influencer", "someone who is famous on tiktok"],
  ["conflict", "drama"],
  ["clothing worn on the feet","socks"],
  ["throughout","always"],
  ["a break", "vacation"]
]
negative = [
  ["ice cream", "fondue"],
  ["warship", "Statue of Liberty"],
  ["birdfeed", "the study of birds"],
  ["staff made of magic", "jazz-fusion"]
]

predictions=[x['score'] for x in CE_model.rank(query='star', documents=positive[0])]
print(predictions)

for x in positive:
  similarity = CE_model.predict(x)
  print(x, ", ", similarity)
for x in negative:
  similarity = CE_model.predict(x)
  print(x, ", ", similarity)

pre-training results:
```
['place of origin', 'birthplace'] ,  0.00043210216
['alien', 'foreigner'] ,  0.0002477122
['warship', 'naval war vessel'] ,  0.01252699
['lemon treats', 'lemon-flavored candy'] ,  0.89636856
['conflict', 'drama'] ,  5.4892906e-05
['clothing worn on the feet', 'socks'] ,  0.0020660206
['throughout', 'always'] ,  3.7185084e-05
['a break', 'vacation'] ,  4.2760672e-05

['ice cream', 'fondue'] ,  3.9480252e-05
['warship', 'Statue of Liberty'] ,  1.4958433e-05
['birdfeed', 'the study of birds'] ,  0.00034959082
['staff made of magic', 'jazz-fusion'] ,  1.2440907e-05
```

## SentenceTransformer Model

In [ ]:
ST_model = SentenceTransformer("all-MiniLM-L6-v2", device="mps" if torch.backends.mps.is_available() else "cpu")

loss = losses.CosineSimilarityLoss(ST_model)
print(ST_model)

ST_model_save_path = "output/training_ST_" + datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

# train the model
ST_model.fit(
    train_objectives=[(
        DataLoader(
            dataset=train_data,
            shuffle=True,
            batch_size=train_batch_size,
            pin_memory=True
        ),
        loss
    )],
    evaluator=evaluation.BinaryClassificationEvaluator.from_input_examples(cv_data),
    epochs=num_epochs,
    warmup_steps=warmup_steps,
    output_path=ST_model_save_path
)

# write the config
with open(ST_model_save_path + "/config.txt", 'w') as f:
    f.write("train batch size: " + str(train_batch_size) + "\n")
    f.write("eval batch size: " + str(eval_batch_size) + "\n")
    f.write("num epochs: " + str(num_epochs) + "\n")
    f.write("warmup_steps: " + str(warmup_steps) + "\n")
    f.write("data_ratio: " + str(1 / data_ratio) + "\n")
    f.write("no dense layer")

In [ ]:
positive = [
  ["place of origin", "birthplace"],
  ["alien", "foreigner"],
  ["warship", "naval war vessel"],
  ["lemon treats", "lemon-flavored candy"],
  ["conflict", "drama"],
  ["clothing worn on the feet","socks"],
  ["throughout","always"],
  ["a break", "vacation"]
]
negative = [
  ["ice cream", "fondue"],
  ["warship", "Statue of Liberty"],
  ["birdfeed", "the study of birds"],
  ["staff made of magic", "jazz-fusion"]
]

for [x, y] in positive:
  similarity = float(ST_model.similarity(ST_model.encode(x), ST_model.encode(y)))
  print([x, y], ", ", similarity)

for [x, y] in negative:
  similarity = float(ST_model.similarity(ST_model.encode(x), ST_model.encode(y)))
  print([x, y], ", ", similarity)


pre-training results:
```
['place of origin', 'birthplace'] ,  0.5736582279205322
['alien', 'foreigner'] ,  0.5416784286499023
['warship', 'naval war vessel'] ,  0.7769221663475037
['lemon treats', 'lemon-flavored candy'] ,  0.7464485764503479
['conflict', 'drama'] ,  0.47645804286003113
['clothing worn on the feet', 'socks'] ,  0.5724813938140869
['throughout', 'always'] ,  0.355945348739624
['a break', 'vacation'] ,  0.41463303565979004

['ice cream', 'fondue'] ,  0.4437253177165985
['warship', 'Statue of Liberty'] ,  0.3803083300590515
['birdfeed', 'the study of birds'] ,  0.5581598281860352
['staff made of magic', 'jazz-fusion'] ,  0.05649913102388382
```